# Phase 3 — Exploratory Data Analysis
Competition: Playground Series S6E7 — Predicting Student Health Risk  
Target: `health_condition` (at-risk / unhealthy / fit)  
Metric: **Balanced accuracy**

In [ ]:
import sys
sys.path.append('..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style='whitegrid')
pd.set_option('display.max_columns', 50)

## 1. Load & first look

In [ ]:
train = pd.read_csv('../data/raw/train.csv')
test  = pd.read_csv('../data/raw/test.csv')

print('Train shape:', train.shape)
print('Test shape: ', test.shape)
train.head()

In [ ]:
train.info()

## 2. Missing values

In [ ]:
missing = train.isnull().sum().sort_values(ascending=False)
missing_pct = (train.isnull().mean() * 100).round(1).sort_values(ascending=False)
pd.concat([missing, missing_pct], axis=1, keys=['count', '%']).head(20)

## 3. Target distribution — critical for balanced_accuracy metric

In [ ]:
print(train['health_condition'].value_counts())
print('\nClass proportions:')
print(train['health_condition'].value_counts(normalize=True).round(3))

fig, ax = plt.subplots(figsize=(6, 4))
train['health_condition'].value_counts().plot(kind='bar', ax=ax, color=['#e74c3c','#f39c12','#2ecc71'])
ax.set_title('Target Distribution — health_condition')
ax.set_xlabel('')
plt.tight_layout()
plt.savefig('../models/target_distribution.png', dpi=100)
plt.show()

**Finding:** Record class balance / imbalance here. If any class < 20% → use `class_weight='balanced'` in all sklearn models.

## 4. Univariate exploration — numeric features

In [ ]:
num_cols = train.select_dtypes(include='number').columns.drop('id', errors='ignore').tolist()
print('Numeric columns:', num_cols)
train[num_cols].describe()

In [ ]:
train[num_cols].hist(figsize=(15, 10), bins=30)
plt.tight_layout()
plt.savefig('../models/numeric_histograms.png', dpi=100)
plt.show()

## 5. Correlation heatmap

In [ ]:
corr = train[num_cols].corr()
plt.figure(figsize=(12, 9))
sns.heatmap(corr, cmap='coolwarm', center=0, annot=True, fmt='.2f', linewidths=0.5)
plt.title('Correlation Matrix — Numeric Features')
plt.tight_layout()
plt.savefig('../models/correlation_heatmap.png', dpi=100)
plt.show()

## 6. Feature vs target (numeric)

In [ ]:
fig, axes = plt.subplots(nrows=(len(num_cols)+2)//3, ncols=3,
                          figsize=(15, 4*((len(num_cols)+2)//3)))
axes = axes.flatten()
for i, col in enumerate(num_cols):
    train.boxplot(column=col, by='health_condition', ax=axes[i])
    axes[i].set_title(col)
    axes[i].set_xlabel('')
for j in range(i+1, len(axes)):
    axes[j].set_visible(False)
plt.suptitle('Numeric Features by Target Class')
plt.tight_layout()
plt.savefig('../models/features_vs_target.png', dpi=100)
plt.show()

## 7. Categorical features

In [ ]:
cat_cols = train.select_dtypes(include='object').columns.drop('health_condition', errors='ignore').tolist()
print('Categorical columns:')
for c in cat_cols:
    print(f'  {c}: {train[c].nunique()} unique values → {train[c].unique()[:5]}')

In [ ]:
for col in cat_cols:
    ct = pd.crosstab(train[col], train['health_condition'], normalize='index').round(3)
    print(f'\n{col}:')
    display(ct)

## 8. Outlier detection (IQR)

In [ ]:
Q1 = train[num_cols].quantile(0.25)
Q3 = train[num_cols].quantile(0.75)
IQR = Q3 - Q1
outlier_count = ((train[num_cols] < (Q1 - 1.5*IQR)) | (train[num_cols] > (Q3 + 1.5*IQR))).sum()
print('Outlier counts per feature:')
print(outlier_count.sort_values(ascending=False))

## 9. EDA Findings Summary

**Fill this in after running the cells above:**

1. **Class balance:** `health_condition` distribution is ________ → will use `class_weight='balanced'`
2. **Missing values:** Columns with >5% missing: ________ → strategy: median/mode impute
3. **Skewed features:** ________ → may clip or log-transform
4. **High correlation:** ________ correlated with ________ → drop one
5. **Categorical cardinality:** All cats have low cardinality → OneHotEncoder is fine